# 📉 Customer Churn Prediction

Predicts whether a telecom customer will **churn (leave)** based on behavioral and demographic data.

**Dataset:** [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — 7,043 customers, 21 features

**Skills covered:** Feature Engineering · Classification · Model Tuning · Evaluation Metrics

| Model | Type |
|-------|------|
| Logistic Regression | Baseline classifier |
| Random Forest | Ensemble, handles non-linearity |
| XGBoost | Best performance, gradient boosting |

---

## 📦 Step 1: Install & Import Libraries

In [ ]:
!pip install kaggle xgboost imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score,
                              roc_curve, ConfusionMatrixDisplay)

# XGBoost
from xgboost import XGBClassifier

# Class imbalance
from imblearn.over_sampling import SMOTE

import pickle
plt.style.use('ggplot')
print('✅ All libraries imported successfully!')

## 📥 Step 2: Download & Load Dataset

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d blastchar/telco-customer-churn

from zipfile import ZipFile
with ZipFile('telco-customer-churn.zip', 'r') as z:
    z.extractall()

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'✅ Dataset loaded! Shape: {df.shape}')
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info:')
print(f'Shape      : {df.shape}')
print(f'Duplicates : {df.duplicated().sum()}')
print(f'\nColumn dtypes:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(churn_counts.index, churn_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Churn Count', fontsize=13)
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=11, fontweight='bold')

axes[1].pie(churn_pct, labels=churn_pct.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Churn Proportion', fontsize=13)

plt.suptitle('Target Variable — Churn Distribution', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Churn Rate: {churn_pct["Yes"]:.1f}%  →  Class Imbalance present!')

In [ ]:
# Numerical features vs Churn
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    df_temp = df.copy()
    df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
    df_temp.dropna(subset=[col], inplace=True)
    ax.hist(df_temp[df_temp['Churn']=='No'][col],  bins=30, alpha=0.6,
            color='#2ecc71', label='No Churn')
    ax.hist(df_temp[df_temp['Churn']=='Yes'][col], bins=30, alpha=0.6,
            color='#e74c3c', label='Churn')
    ax.set_title(f'{col} by Churn', fontsize=12)
    ax.set_xlabel(col)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs Churn rate
cat_cols = ['Contract', 'InternetService', 'PaymentMethod',
            'TechSupport', 'OnlineSecurity', 'PaperlessBilling']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').mean() * 100
    ).sort_values(ascending=False)
    bars = ax.bar(churn_rate.index, churn_rate.values,
                  color='#e74c3c', edgecolor='black', alpha=0.8)
    ax.set_title(f'Churn Rate by {col}', fontsize=11)
    ax.set_ylabel('Churn Rate (%)')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontsize=9)

plt.suptitle('Churn Rate by Categorical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 🔧 Step 4: Feature Engineering & Preprocessing

In [ ]:
df_model = df.copy()

# Drop customerID — not a feature
df_model.drop(columns=['customerID'], inplace=True)

# Fix TotalCharges — has spaces, should be numeric
df_model['TotalCharges'] = pd.to_numeric(df_model['TotalCharges'], errors='coerce')
df_model['TotalCharges'].fillna(df_model['TotalCharges'].median(), inplace=True)

# ── Feature Engineering ───────────────────────────────────────────
# Avg monthly spend
df_model['AvgMonthlySpend'] = df_model['TotalCharges'] / (df_model['tenure'] + 1)

# Tenure group buckets
df_model['TenureGroup'] = pd.cut(df_model['tenure'],
                                  bins=[0, 12, 24, 48, 72],
                                  labels=['0-1yr', '1-2yr', '2-4yr', '4+yr'])

# Number of services subscribed
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df_model['NumServices'] = df_model[service_cols].apply(
    lambda row: sum(1 for v in row if v not in ['No', 'No internet service',
                                                  'No phone service']), axis=1
)

print('✅ Feature engineering complete!')
print(f'New features added: AvgMonthlySpend, TenureGroup, NumServices')
df_model[['tenure', 'TenureGroup', 'TotalCharges',
           'AvgMonthlySpend', 'NumServices']].head()

In [ ]:
# Encode target
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

# Encode all remaining categorical columns with Label Encoding
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print('✅ Encoding complete!')
print(f'Encoded columns: {cat_cols}')
print(f'Final shape: {df_model.shape}')

In [ ]:
# Correlation with Churn
corr = df_model.corr()['Churn'].sort_values(ascending=False)
plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in corr[1:]]
plt.barh(corr[1:].index, corr[1:].values, color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Churn', fontsize=13)
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

## ✂️ Step 5: Train / Test Split + Handle Class Imbalance (SMOTE)

In [ ]:
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE — fix class imbalance in training set only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'Before SMOTE → Churn: {y_train.sum()} | No Churn: {(y_train==0).sum()}')
print(f'After  SMOTE → Churn: {y_train_resampled.sum()} | No Churn: {(y_train_resampled==0).sum()}')

## 📈 Step 6: Train All 3 Models

In [ ]:
# ── Logistic Regression ───────────────────────────────────────────
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)
print('✅ Logistic Regression trained!')

# ── Random Forest ─────────────────────────────────────────────────
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_resampled, y_train_resampled)
print('✅ Random Forest trained!')

# ── XGBoost ───────────────────────────────────────────────────────
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1,
                            max_depth=5, random_state=42,
                            use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train_resampled, y_train_resampled)
print('✅ XGBoost trained!')

## 📊 Step 7: Evaluate All Models

In [ ]:
def evaluate_classifier(name, model, X_test, y_test):
    y_pred     = model.predict(X_test)
    y_prob     = model.predict_proba(X_test)[:, 1]
    acc        = accuracy_score(y_test, y_pred)
    roc_auc    = roc_auc_score(y_test, y_prob)
    print(f'─── {name} ───')
    print(f'Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
    print(f'ROC-AUC   : {roc_auc:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    return {'Model': name, 'Accuracy': acc, 'ROC-AUC': roc_auc,
            'y_pred': y_pred, 'y_prob': y_prob}

lr_results  = evaluate_classifier('Logistic Regression', lr_model,  X_test_scaled, y_test)
rf_results  = evaluate_classifier('Random Forest',       rf_model,  X_test_scaled, y_test)
xgb_results = evaluate_classifier('XGBoost',             xgb_model, X_test_scaled, y_test)

In [ ]:
# Confusion matrices — all 3 models
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, res in zip(axes, [lr_results, rf_results, xgb_results]):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    ax.set_title(res['Model'], fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — All Models', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))
for res, color in zip([lr_results, rf_results, xgb_results],
                       ['steelblue', 'darkorange', 'green']):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f'{res["Model"]} (AUC = {res["ROC-AUC"]:.3f})')
plt.plot([0,1],[0,1],'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison bar chart
summary = pd.DataFrame([
    {'Model': r['Model'], 'Accuracy': r['Accuracy'], 'ROC-AUC': r['ROC-AUC']}
    for r in [lr_results, rf_results, xgb_results]
])

summary_melt = summary.melt(id_vars='Model', var_name='Metric', value_name='Score')
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=summary_melt, x='Model', y='Score', hue='Metric',
                 palette=['steelblue', 'darkorange'])
plt.ylim(0.5, 1.0)
plt.title('Model Comparison — Accuracy & ROC-AUC', fontsize=13)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.3f}',
                (p.get_x() + p.get_width()/2, p.get_height() + 0.003),
                ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 🔧 Step 8: Hyperparameter Tuning (XGBoost)

In [ ]:
# GridSearchCV on XGBoost — best model
param_grid = {
    'n_estimators'  : [100, 200],
    'max_depth'     : [3, 5, 7],
    'learning_rate' : [0.05, 0.1, 0.2],
}

xgb_tuned = XGBClassifier(random_state=42, use_label_encoder=False,
                            eval_metric='logloss')
grid_search = GridSearchCV(xgb_tuned, param_grid,
                            cv=3, scoring='roc_auc',
                            n_jobs=-1, verbose=1)
grid_search.fit(X_train_resampled, y_train_resampled)

print(f'\n✅ Best Parameters : {grid_search.best_params_}')
print(f'Best CV ROC-AUC    : {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate tuned model
best_xgb = grid_search.best_estimator_
tuned_results = evaluate_classifier('XGBoost (Tuned)', best_xgb, X_test_scaled, y_test)

## 🌟 Step 9: Feature Importance (XGBoost)

In [ ]:
feat_imp = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': best_xgb.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='Reds_r')
plt.title('Top 15 Feature Importances — XGBoost', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print(feat_imp.to_string(index=False))

## 🔮 Step 10: Predict on a Custom Customer

In [ ]:
def predict_churn(customer_df):
    """
    Pass a single-row DataFrame with the same columns as X.
    Returns churn probability and prediction.
    """
    scaled = scaler.transform(customer_df)
    prob   = best_xgb.predict_proba(scaled)[0][1]
    label  = 'Churn ❌' if prob >= 0.5 else 'No Churn ✅'
    print(f'Churn Probability : {prob:.2%}')
    print(f'Prediction        : {label}')
    return prob

# Example customer — use actual column values from your encoded df
sample = X_test.iloc[[0]]
print('Sample customer data:')
print(sample.T)
print()
predict_churn(sample)

## 💾 Step 11: Save Best Model

In [ ]:
with open('churn_xgb_model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)
with open('churn_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('✅ Model saved  → churn_xgb_model.pkl')
print('✅ Scaler saved → churn_scaler.pkl')

---
## 📋 Summary

| Step | What was done |
|------|---------------|
| EDA | Churn distribution, numerical & categorical analysis |
| Feature Engineering | AvgMonthlySpend, TenureGroup, NumServices |
| Preprocessing | Label encoding, StandardScaler, SMOTE for class imbalance |
| Models | Logistic Regression, Random Forest, XGBoost |
| Tuning | GridSearchCV on XGBoost (n_estimators, max_depth, learning_rate) |
| Metrics | Accuracy, ROC-AUC, Confusion Matrix, Classification Report |
| Output | Saved best model + scaler as .pkl files |

**Key Insight:** `tenure`, `Contract type`, `MonthlyCharges`, and `TechSupport` are the strongest predictors of churn.

Customers on **month-to-month contracts** with **high monthly charges** and **short tenure** are most likely to churn.